# 主性能 · 共享召回池与回答模型
HotpotQA / MuSiQue / 2Wiki 跨数据集；保存逐题原始轨迹和配对统计。

In [5]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'full'  # Change to 'full' only after the pilot completes and memory is measured.
ROOT = DRIVE / 'public_lightrag_v1' / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
PUBLIC_ASSETS = DRIVE / 'public_lightrag_assets'
CONFIG_PATH = PROJECT / 'configs/public_lightrag.yaml'
print('Code:', PROJECT, 'Artifacts:', ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Code: /content/graphrag-jev Artifacts: /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full


In [6]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    # BF16 + LoRA does not use torchao quantization. Colab's old optional
    # torchao package makes PEFT 0.19.1 fail even on ordinary Linear layers.
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# Activate editable-install .pth files in this already-running notebook kernel.
import importlib, site
site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())
for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)
importlib.invalidate_caches()
import sa_jev
print('SA-Jev source:', sa_jev.__file__)
# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config(CONFIG_PATH)
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    from collections import deque
    tail = deque(maxlen=40)
    cmd = [sys.executable, '-u', '-m', 'sa_jev.cli', command, '--root', str(ROOT), '--config', str(CONFIG_PATH), *map(str,args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f'sajev {command} failed (exit {returncode}). Last output: ' + ''.join(tail))


SA-Jev source: /content/graphrag-jev/src/sa_jev/__init__.py


In [7]:
# Read the same credential file in every notebook. Fill it in once.
from sa_jev.environment import load_environment
ENV_FILE = (DRIVE if 'google.colab' in sys.modules else PROJECT) / '.env'
credential_status = load_environment(ENV_FILE, template=PROJECT / '.env.example')
print('Credential file:', ENV_FILE)
for name in ('OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY', 'HF_TOKEN'):
    print(name + ':', 'loaded' if credential_status[name] else 'empty')
if not credential_status['OPENAI_API_KEY']:
    print('Fill OPENAI_API_KEY in the file above, save it, then rerun this cell before building the index or answering questions.')


Credential file: /content/drive/MyDrive/SA-Jev/.env
OPENAI_API_KEY: loaded
OPENAI_BASE_URL: loaded
TYPESAFE_API_KEY: loaded
HF_TOKEN: loaded


In [8]:
from sa_jev.experiments import main_experiment
rows = main_experiment(ROOT, config, seeds=[17] if MODE=='pilot' else config['seeds'], limit=8 if MODE=='pilot' else None)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

[OpenJev] Checking artifact: /content/drive/MyDrive/SA-Jev/huggingface/hub/models--shenjunhao--OpenJev-4B/snapshots/064bd435c6713486450194e29d526266b28941e2
[OpenJev] Loading verified ordinary files from local runtime: /tmp/sajev_openjev_models/fd2ba2d05d9eb7ca5bbc3411
[OpenJev] 1/11 backbone/config.json: expected 1,977 bytes; checking SHA-256...
[OpenJev] 2/11 backbone/model-00001-of-00003.safetensors: expected 3,991,295,368 bytes; checking SHA-256...
[OpenJev] 3/11 backbone/model-00002-of-00003.safetensors: expected 3,979,828,128 bytes; checking SHA-256...
[OpenJev] 4/11 backbone/model-00003-of-00003.safetensors: expected 440,425,856 bytes; checking SHA-256...
[OpenJev] 5/11 backbone/model.safetensors.index.json: expected 33,047 bytes; checking SHA-256...
[OpenJev] 6/11 calibration.json: expected 44 bytes; checking SHA-256...
[OpenJev] 7/11 decision_head.safetensors: expected 37,809,208 bytes; checking SHA-256...
[OpenJev] 8/11 openjev_config.json: expected 458 bytes; checking SHA-25

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


[ERROR] `loss` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.
[ERROR] `logits` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

[OpenJev] Load backbone/head/tokenizer; target=cuda, dtype=bfloat16: exited after 12.4s
[OpenJev] Model loaded on cuda; configuring decision branches
[OpenJev] Delta kernel=fla.ops.gated_delta_rule.chunk.chunk_gated_delta_rule; convolution=torch
[OpenJev] branch_microbatch=1, independent decision batch=8
retrieval_order 1/1100 status=ok ETA=6032.3s peak_reserved_gib=8.046875
retrieval_order 2/1100 status=ok ETA=3482.5s peak_reserved_gib=8.111328125
retrieval_order 3/1100 status=ok ETA=3192.3s peak_reserved_gib=8.111328125
retrieval_order 4/1100 status=ok ETA=2847.9s peak_reserved_gib=8.111328125
retrieval_order 5/1100 status=ok ETA=2925.5s peak_reserved_gib=8.111328125
retrieval_order 6/1100 status=ok ETA=2591.2s peak_reserved_gib=8.111328125
retrieval_order 7/1100 status=ok ETA=2352.2s peak_reserved_gib=8.111328125
retrieval_order 8/1100 status=ok ETA=2352.0s peak_reserved_gib=8.111328125
retrieval_order 9/1100 status=ok ETA=2192.4s peak_reserved_gib=8.111328125
retrieval_order 10/110

KeyboardInterrupt: 

候选缓存相同、预算相同、回答器相同。原始召回排序使用冻结 OpenJev 的独立充分性头。Jev API 缺失会写 unavailable.json，不伪造可比较结果。未知标签的指标留空并报告有效样本数。